# BUS32120, Final Project

# SQL Exercises on Education and Income Data

To run this file, you'll need access to the raw data files included in the GitHub repository: 
1. 10 years of education data (setup in this file)
2. Income data

In [1]:
import sqlite3 as sq
import pandas as pd

### 0.1) Create new database, open connection to it, and create new (empty table)

In [2]:
# create a connection and (if it doesn't exist) a new db
connection = sq.connect('education.db') 

In [3]:
# a cursor to be used with the connection
cursor = connection.cursor()

In [4]:
#Drops education data table if it exists
query = """DROP TABLE IF EXISTS education_data;"""

cursor.execute(query)

In [5]:
#Drops wage data table if it exists
query = """DROP TABLE IF EXISTS wage_data;"""

cursor.execute(query)

In [6]:
#Creates the education data table with the following columns setup to receive the data
query = """CREATE TABLE education_data(
  id INT,
  School TEXT,
  State TEXT,
  Predominantly_Undergraduate_Institution INT,
  Ownership INT,
  In_State_Tuition REAL,
  Out_of_State_Tuition REAL,
  Federal_Loan_Rate REAL,
  Median_Debt_All_Income_Brackets REAL,
  Median_Debt_0_30000_Income REAL,
  Median_Debt_30001_75000_Income REAL,
  Median_Debt_75000_plus_Income REAL,
  Number_of_Undergrads_Enrolled REAL,
  Year INT
);
"""

cursor.execute(query)

In [7]:
#Creates the wage data table with the following columns setup to receive the data
query = """CREATE TABLE wage_data(
  State TEXT,
  A_MEDIAN REAL,
  Year INT
);
"""

cursor.execute(query)

### 0.2) Read in education data csv and put that csv into the SQL db created above

In [8]:
#List of columns to keep from the education data
cols_to_keep = [
    "UNITID",
    "INSTNM",
    "STABBR",
    "CONTROL",
    "UG12MN",
    "PCTFLOAN",
    "LO_INC_DEBT_MDN",
    "MD_INC_DEBT_MDN",
    "HI_INC_DEBT_MDN",
    "DEBT_MDN",
    "TUITIONFEE_IN",
    "TUITIONFEE_OUT",
    "PREDDEG"
]

#List of columns with number values
numeric_cols = [
    "Ownership",
    "Number_of_Undergrads_Enrolled",
    "Federal_Loan_Rate",
    "Median_Debt_0_30000_Income",
    "Median_Debt_30001_75000_Income",
    "Median_Debt_75000_plus_Income",
    "Median_Debt_All_Income_Brackets",
    "In_State_Tuition",
    "Out_of_State_Tuition",
    "Predominantly_Undergraduate_Institution"
]

#Creates an empty df for files to merge into
education_data_merged = []

#Loop that will pull the education data file from 2013 to 2023
for year in range(2013, 2023):
    file = f"MERGED{year}_{str(year + 1)[-2:]}_PP.csv"
    
    edu_frame = pd.read_csv(file, usecols=cols_to_keep, low_memory=False)
   
    #Renames the columns to more logical values
    edu_small = edu_frame.rename(columns={
        "UNITID": "id",
        "INSTNM": "School",
        "STABBR": "State",
        "CONTROL": "Ownership",
        "UG12MN": "Number_of_Undergrads_Enrolled",
        "PCTFLOAN": "Federal_Loan_Rate",
        "LO_INC_DEBT_MDN": "Median_Debt_0_30000_Income",
        "MD_INC_DEBT_MDN": "Median_Debt_30001_75000_Income",
        "HI_INC_DEBT_MDN": "Median_Debt_75000_plus_Income",
        "DEBT_MDN": "Median_Debt_All_Income_Brackets",
        "TUITIONFEE_IN": "In_State_Tuition",
        "TUITIONFEE_OUT": "Out_of_State_Tuition",
        "PREDDEG": "Predominantly_Undergraduate_Institution"
    })

    #Creates a new column with the year
    edu_small["Year"] = year

    #Filters out the prive and non undergrad institutions from the education data
    edu_small = edu_small[edu_small["Predominantly_Undergraduate_Institution"] == 3]
    
    # Attempt to convert columns to numeric where possible.
    for col in numeric_cols:
        edu_small[col] = pd.to_numeric(edu_small[col], errors="coerce")
    
    # Calculate the average median debt for students from households with income $0-$30,000
    low_avg = edu_small["Median_Debt_0_30000_Income"].mean()
    
    # Replace missing values in the low-income debt column with the column average
    edu_small["Median_Debt_0_30000_Income"] = edu_small["Median_Debt_0_30000_Income"].fillna(low_avg)
    
    # Calculate the average median debt for students from households with income $30,001-$75,000
    medium_avg = edu_small["Median_Debt_30001_75000_Income"].mean()
    
    # Replace missing values in the middle-income debt column with the column average
    edu_small["Median_Debt_30001_75000_Income"] = edu_small["Median_Debt_30001_75000_Income"].fillna(medium_avg)
    
    
    # Calculate the average median debt for students from households with income greater than $75,000
    high_avg = edu_small["Median_Debt_75000_plus_Income"].mean()
    
    # Replace missing values in the high-income debt column with the column average
    edu_small["Median_Debt_75000_plus_Income"] = edu_small["Median_Debt_75000_plus_Income"].fillna(high_avg)
    
    
    # Calculate the overall average median debt across all income brackets
    edu_small = edu_small.dropna(subset=["Median_Debt_All_Income_Brackets"])
    all_avg = edu_small["Median_Debt_All_Income_Brackets"].mean()
    
    # Replace missing values in the overall debt column with the column average
    edu_small["Median_Debt_All_Income_Brackets"] = edu_small["Median_Debt_All_Income_Brackets"].fillna(all_avg)
    
    
    # Remove rows where key financial or enrollment data is missing
    edu_small = edu_small.dropna(subset=[
        "Number_of_Undergrads_Enrolled",
        "Federal_Loan_Rate",
        "In_State_Tuition",
        "Out_of_State_Tuition"
    ])

    # Calculating the amount of total student loan debt carried by each university's students
    edu_small["University_Debt_Load"] = (
        edu_small["Number_of_Undergrads_Enrolled"]
        * edu_small["Median_Debt_All_Income_Brackets"]
        * edu_small["Federal_Loan_Rate"]
    )
    
    
    # Round  values and convert them to integers
    edu_small["Number_of_Undergrads_Enrolled"] = edu_small["Number_of_Undergrads_Enrolled"].round().astype("int64")
    edu_small["Median_Debt_All_Income_Brackets"] = edu_small["Median_Debt_All_Income_Brackets"].round().astype("int64")
    
    
    # Calculate the total university debt load:
    # (number of undergraduates) × (median debt at graduation) × (federal loan participation rate)
    edu_small["University_Debt_Load"] = (
        edu_small["Number_of_Undergrads_Enrolled"]
        * edu_small["Median_Debt_All_Income_Brackets"]
        * edu_small["Federal_Loan_Rate"]
    ) / 1000000

    # Convert federal loan rate (0–1) to a formatted percentage string
    edu_small["Percent_with_Federal_Loans"] = edu_small["Federal_Loan_Rate"].apply(
        lambda x: f"{x * 100:.2f}%"
    )
    #appends the temp df into the aggregated dataset
    education_data_merged.append(edu_small)

#Push this into a dataframe
edu_frame = pd.concat(education_data_merged, ignore_index=True)

In [9]:
edu_frame.head()

,id,School,State,Predominantly_Undergraduate_Institution,Ownership,In_State_Tuition,Out_of_State_Tuition,Federal_Loan_Rate,Median_Debt_All_Income_Brackets,Median_Debt_0_30000_Income,Median_Debt_30001_75000_Income,Median_Debt_75000_plus_Income,Number_of_Undergrads_Enrolled,Year,University_Debt_Load,Percent_with_Federal_Loans
0,100654,Alabama A & M University,AL,3,1,7182.0,12774.0,0.8204,12430,11500.0,14250.000000,15250.000000,4468,2013,45.562752,82.04%
1,100663,University of Alabama at Birmingham,AL,3,1,7206.0,16398.0,0.5397,12500,12500.0,12500.000000,11250.000000,13108,2013,88.429845,53.97%
2,100690,Amridge University,AL,3,2,6870.0,6870.0,0.7629,7376,7375.5,15535.991774,15068.267609,484,2013,2.723541,76.29%
3,100706,University of Alabama in Huntsville,AL,3,1,9192.0,21506.0,0.4728,13000,12500.0,13000.000000,13000.000000,6695,2013,41.150148,47.28%
4,100724,Alabama State University,AL,3,1,8720.0,15656.0,0.8735,11000,10630.5,12250.000000,11772.000000,5906,2013,56.747801,87.35%


In [10]:
#Checks the output of the table just created
edu_frame.info()

<class 'pandas.DataFrame'>
RangeIndex: 11986 entries, 0 to 11985
Data columns (total 16 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   id                                       11986 non-null  int64  
 1   School                                   11986 non-null  str    
 2   State                                    11986 non-null  str    
 3   Predominantly_Undergraduate_Institution  11986 non-null  int64  
 4   Ownership                                11986 non-null  int64  
 5   In_State_Tuition                         11986 non-null  float64
 6   Out_of_State_Tuition                     11986 non-null  float64
 7   Federal_Loan_Rate                        11986 non-null  float64
 8   Median_Debt_All_Income_Brackets          11986 non-null  int64  
 9   Median_Debt_0_30000_Income               11986 non-null  float64
 10  Median_Debt_30001_75000_Income           11986 non-null  

In [11]:
#Finally pushes this dataframe up to the sql server and into the education_data table
edu_frame.to_sql('education_data', connection, if_exists='replace', index=False)

11986

### 0.3) Read in wage data xlsx and put that xlsx into the SQL db created above

In [12]:
#Creates an empty list for the wage data to fill into later
all_frames = []

#Loop that will pull the wage data collected for each year
for year in range(2014, 2024):
    file = f"state_M{year}_dl.xlsx"
    state = pd.read_excel(file)

    # standardize column names for checking
    cols = list(state.columns)

#Not all of the data was able to be loaded in the same way to is needed to be passed through an if statement to pull them in correctly
    if year in range(2014, 2019):
        # 2014–2018 schema
        state = state.loc[
            state["OCC_TITLE"] == "All Occupations",
            ["ST", "A_MEDIAN"]
        ].copy()

        #Renames the columns
        state = state.rename(columns={
            "ST": "State",
            "A_MEDIAN": "A_MEDIAN"
        })

    elif year == 2019:
        # 2019 schema
        state = state.loc[
            state["occ_title"] == "All Occupations",
            ["area_title", "a_median"]
        ].copy()

        state = state.rename(columns={
            "area_title": "State",
            "a_median": "A_MEDIAN"
        })

    else:
        # 2020–2023 schema
        state.columns = state.columns.str.strip()

        state = state.loc[
            (state["OCC_TITLE"] == "All Occupations") & (state["AREA_TYPE"] == 2),
            ["PRIM_STATE", "A_MEDIAN"]
        ].copy()

        state = state.rename(columns={
            "PRIM_STATE": "State",
            "A_MEDIAN": "A_MEDIAN"
        })

    # clean median wage column of the commas and spaces
    state["A_MEDIAN"] = (
        state["A_MEDIAN"]
        .astype(str)
        .str.replace(",", "", regex=False)
    )

    state["A_MEDIAN"] = pd.to_numeric(state["A_MEDIAN"], errors="coerce")

    # add year column
    state["Year"] = year

    #Appends the temporary dataframe into the all_frames dataset
    all_frames.append(state)
#Attaches all of these onto the consolidated dataframe
state_wages = pd.concat(all_frames, ignore_index=True)

state_wages.head()

,State,A_MEDIAN,Year
0,AL,30850,2014
1,AK,45200,2014
2,AZ,34230,2014
3,AR,29140,2014
4,CA,39190,2014


In [13]:
#Finally pushes this up to the wage_data table
state_wages.to_sql('wage_data', connection, if_exists='replace', index=False)

528

## Start of SQL Analysis



In [14]:
#This query calculates the average in-state tution from the education data and breaks it out by year
query = """ SELECT Year, AVG(In_State_Tuition) AS Avg_In_State_Tuition
                FROM education_data
                    GROUP BY Year
                        ORDER BY Year; """

output = pd.read_sql(query, connection) 
output

,Year,Avg_In_State_Tuition
0,2013,19589.908765
1,2014,20195.500522
2,2015,21076.180415
3,2016,22298.829268
4,2017,22371.174356
5,2018,24982.121524
6,2019,25458.889792
7,2020,24337.569327


##### Results show tuition generally increased from about \\$19.6K in 2013 to about \\$25.5K in 2019, indicating rising college costs over time.

In [15]:
query = """ SELECT Year, AVG(Out_of_State_Tuition) AS Avg_Out_of_State_Tuition
                FROM education_data
                    GROUP BY Year
                        ORDER BY Year; """

output = pd.read_sql(query, connection) 
output

,Year,Avg_Out_of_State_Tuition
0,2013,22523.002563
1,2014,23320.096134
2,2015,24367.383183
3,2016,26175.392120
4,2017,25849.818670
5,2018,28647.760041
6,2019,29177.248419
7,2020,27993.113186


##### Results show out-of-state tuition generally increased from about \\$16.3K in 2013 to about \\$21.6K in 2022, indicating rising college costs over time.

In [16]:
#This query calculates the median wage averaged across all of the state from the wage data and breaks it out by year
query = """ SELECT Year, AVG(A_median) AS Median_Salary
                FROM wage_data
                    GROUP BY Year
                        ORDER BY Year; """

output = pd.read_sql(query, connection) 
output

,Year,Median_Salary
0,2014,34966.481481
1,2015,35655.555556
2,2016,36513.888889
3,2017,37289.444444
4,2018,38322.407407
5,2019,39610.740741
6,2020,42432.549020
7,2021,43322.549020
8,2022,45725.490196
9,2023,48486.274510


##### Results show wages generally increased from about \\$35K in 2014 to about \\$48.5K in 2023, indicating rising wages over time.

In [17]:
#Joins education_data and wage_data on State and Year to align tuition data with state wage data.
query = """ SELECT e.Year, AVG(e.In_State_Tuition) AS Avg_In_State_Tuition, AVG(w.A_MEDIAN) AS Avg_State_Wage
                FROM education_data e
                    JOIN wage_data w ON e.State = w.State AND e.Year = w.Year
                        GROUP BY e.Year
                            ORDER BY e.Year; """

output = pd.read_sql(query, connection)
output

,Year,Avg_In_State_Tuition,Avg_State_Wage
0,2014,20195.500522,35340.271682
1,2015,21076.180415,36086.679085
2,2016,22298.829268,36844.352720
3,2017,22371.174356,37662.317597
4,2018,24982.121524,40149.320288
5,2020,24825.723998,42644.468332


#### Average in-state tuition increased faster than average state wages over the period observed. From 2014 to 2020, tuition rose from about \\$20.2K to \\$24.8K (roughly a 23% increase), while average wages increased from about \\$35.3K to \\$42.6K (about a 21% increase). Although wages grew over time, the slightly faster rise in tuition suggests that college affordability may be gradually worsening, potentially increasing the financial burden on students and families.

In [18]:
# Joins education_data with wage_data on State and Year to align enrollment data with local wage levels.
# Calculates the average number of undergraduates enrolled per state each year and the corresponding average state wage.
query = """ SELECT e.State, e.Year, AVG(e.Number_of_Undergrads_Enrolled) AS Avg_Enrollment, AVG(w.A_MEDIAN) AS Avg_State_Wage
                FROM education_data e
                    JOIN wage_data w ON e.State = w.State AND e.Year = w.Year
                        GROUP BY e.State, e.Year
                            ORDER BY e.State, e.Year; """

output = pd.read_sql(query, connection)
output

,State,Year,Avg_Enrollment,Avg_State_Wage
0,AK,2014,13342.000000,45200.0
1,AK,2015,11101.500000,46420.0
2,AK,2016,12715.666667,47170.0
3,AK,2017,9977.500000,47560.0
4,AK,2018,11576.666667,48020.0
...,...,...,...,...
261,WV,2020,6900.400000,35500.0
262,WY,2014,11509.000000,37770.0
263,WY,2015,11305.000000,38280.0
264,WY,2017,10874.000000,39120.0


#### Results allow comparison of how enrollment levels vary across states relative to economic conditions.

In [19]:
#Joins education_data and wage_data on State and Year to compare tuition with local wage levels.
#Calculates average in-state tuition, average state wage, and the ratio of tuition to wages for each year.
query = """ SELECT e.Year, AVG(e.In_State_Tuition) AS Avg_In_State_Tuition, AVG(w.A_MEDIAN) AS Avg_State_Wage, AVG(e.In_State_Tuition) / AVG(w.A_MEDIAN) AS Tuition_to_Wage_Ratio
                FROM education_data e
                    JOIN wage_data w ON e.State = w.State AND e.Year = w.Year
                        GROUP BY e.Year
                            ORDER BY e.Year; """

output = pd.read_sql(query, connection)
output

,Year,Avg_In_State_Tuition,Avg_State_Wage,Tuition_to_Wage_Ratio
0,2014,20195.500522,35340.271682,0.571459
1,2015,21076.180415,36086.679085,0.584043
2,2016,22298.829268,36844.352720,0.605217
3,2017,22371.174356,37662.317597,0.593994
4,2018,24982.121524,40149.320288,0.622230
5,2020,24825.723998,42644.468332,0.582156


#### The tuition-to-wage ratio increases from 0.57 in 2014 to a peak of 0.62 in 2018, indicating that tuition costs were rising faster than wages during this period and making college less affordable. Although the ratio decreases slightly by 2020, tuition still represents over half of the average annual state wage, highlighting the substantial financial burden higher education can place on families.

In [20]:
#Uses a window function (RANK) to rank states by median wage within each year.
#PARTITION BY Year resets the ranking annually, and higher wages receive a higher rank.
query = """SELECT Year, State, A_MEDIAN, RANK() OVER (PARTITION BY Year ORDER BY A_MEDIAN DESC) AS Wage_Rank
                FROM wage_data
                    ORDER BY Year, Wage_Rank;"""

output = pd.read_sql(query, connection)
output

,Year,State,A_MEDIAN,Wage_Rank
0,2014,DC,64890,1
1,2014,AK,45200,2
2,2014,MA,44670,3
3,2014,CT,42990,4
4,2014,WA,41090,5
...,...,...,...,...
523,2023,AL,41350,47
524,2023,LA,41320,48
525,2023,WV,39770,49
526,2023,AR,39060,50


#### The results show that states like DC, Alaska, and Massachusetts consistently rank among the highest in median wages, while states such as Mississippi, Arkansas, and West Virginia frequently rank near the bottom. This persistent wage disparity suggests that families in lower-income states may face a greater financial burden when paying for college tuition relative to their earnings.

In [21]:
#Uses a window function (RANK) partitioned by year to rank states from highest to lowest tuition.
#This helps identify which states have the most expensive and least expensive public tuition each year.
query = """SELECT Year, State, AVG(In_State_Tuition) AS Avg_In_State_Tuition,
            RANK() OVER (PARTITION BY Year
                ORDER BY AVG(In_State_Tuition) DESC) AS Tuition_Rank
                FROM education_data
                    WHERE In_State_Tuition IS NOT NULL
                        GROUP BY Year, State
                            ORDER BY Year, Tuition_Rank;"""
output = pd.read_sql(query, connection)
output

,Year,State,Avg_In_State_Tuition,Tuition_Rank
0,2013,RI,31461.000000,1
1,2013,MA,29790.108108,2
2,2013,VT,26363.562500,3
3,2013,NY,25939.000000,4
4,2013,CT,25047.730769,5
...,...,...,...,...
355,2020,UT,11580.916667,50
356,2020,PR,6098.477273,51
357,2020,GU,5846.000000,52
358,2020,WY,5791.000000,53


#### States in the Northeast (such as Rhode Island, Massachusetts, and Vermont) consistently rank among the highest for average in-state tuition, while states and territories such as Wyoming, Puerto Rico, and Guam tend to rank among the lowest. This highlights substantial geographic variation in tuition costs, suggesting that where a student attends college can significantly affect the financial burden faced by families.

In [22]:
#Uses a subquery to calculate the average in-state tuition for each year.
#Returns schools whose tuition is higher than the average tuition in that same year.
query = """SELECT School, State, Year, In_State_Tuition
                FROM education_data e
                    WHERE In_State_Tuition > (
                    SELECT AVG(In_State_Tuition)
                            FROM education_data
                                WHERE Year = e.Year
                    )
                        ORDER BY Year, In_State_Tuition DESC;"""

output = pd.read_sql(query, connection)
output

,School,State,Year,In_State_Tuition
0,Columbia University in the City of New York,NY,2013,49138.0
1,Sarah Lawrence College,NY,2013,48696.0
2,Vassar College,NY,2013,47890.0
3,Carnegie Mellon University,PA,2013,47642.0
4,University of Chicago,IL,2013,47514.0
...,...,...,...,...
5376,Saint Leo University,FL,2020,24640.0
5377,Thomas More College of Liberal Arts,NH,2020,24600.0
5378,Hilbert College,NY,2020,24530.0
5379,Hiram College,OH,2020,24500.0


#### The results show that many private and highly selective institutions (such as Columbia University, Vassar College, and the University of Chicago) consistently charge tuition well above the yearly national average. This indicates that certain institutions operate at a significantly higher price point, which may contribute to greater student loan debt for families attending these schools.

In [23]:
#Calculates the average median student debt for low-income (<$30K) and high-income (>$75K) families by state and year.
#Also computes the difference between the two groups (Debt_Gap) to measure inequality in student debt burden.
query = """
SELECT Year, State, 
    AVG(Median_Debt_0_30000_Income) AS Avg_Low_Income_Debt,
    AVG(Median_Debt_75000_plus_Income) AS Avg_High_Income_Debt,
    AVG(Median_Debt_0_30000_Income) - AVG(Median_Debt_75000_plus_Income) AS Debt_Gap
        FROM education_data
            GROUP BY Year, State
                ORDER BY Year; """

output = pd.read_sql(query, connection)
output

,Year,State,Avg_Low_Income_Debt,Avg_High_Income_Debt,Debt_Gap
0,2013,AK,8855.125000,7732.375000,1122.750000
1,2013,AL,11879.648068,13477.190995,-1597.542927
2,2013,AR,10000.479167,12295.980634,-2295.501467
3,2013,AZ,11574.473055,13749.984090,-2175.511036
4,2013,CA,14301.803576,15600.420479,-1298.616903
...,...,...,...,...,...
355,2020,VT,15904.500000,17272.967546,-1368.467546
356,2020,WA,16291.169989,15734.520528,556.649461
357,2020,WI,16493.789474,17737.447368,-1243.657895
358,2020,WV,13042.331243,16005.366886,-2963.035643


#### In many states the Debt_Gap is negative, meaning students from higher-income families often graduate with slightly more debt than lower-income students. This may reflect that higher-income students attend more expensive institutions or borrow differently, while lower-income students may receive more need-based financial aid. However, the size of the gap varies by state, highlighting differences in affordability and financial aid structures across regions.

In [24]:
#Uses a window function (RANK) to rank schools by median student debt within each year.
#PARTITION BY Year resets the ranking annually and orders schools from highest to lowest debt levels.
query = """SELECT School, State, Year, Median_Debt_All_Income_Brackets,
            RANK() OVER (
                PARTITION BY Year
                ORDER BY Median_Debt_All_Income_Brackets DESC
            ) AS Debt_Rank
                FROM education_data
                    WHERE Median_Debt_All_Income_Brackets IS NOT NULL
                        ORDER BY Year, Debt_Rank; """

output = pd.read_sql(query, connection)
output

,School,State,Year,Median_Debt_All_Income_Brackets,Debt_Rank
0,Platt College-Aurora,CO,2013,38885,1
1,Southern California Institute of Architecture,CA,2013,37500,2
2,Southwest University of Visual Arts-Tucson,AZ,2013,31000,3
3,Southwest University of Visual Arts-Albuquerque,NM,2013,31000,3
4,Art Center College of Design,CA,2013,29811,5
...,...,...,...,...,...
11981,University of Puerto Rico-Carolina,PR,2020,4500,1761
11982,CEM College-San Juan,PR,2020,3700,1764
11983,CEM College-Humacao,PR,2020,3700,1764
11984,Berea College,KY,2020,3516,1766


#### The schools with the highest student debt levels are often small private or specialized institutions, particularly in fields such as art and design. In contrast, institutions with the lowest debt levels include schools with strong financial aid models or lower tuition costs, such as Berea College. This highlights how institutional pricing and financial aid policies can significantly influence student debt outcomes.

# SQL Conclusion
This analysis digs into the relationship of rising tuition levels and wages by state over about a 10 year period. The results show that tuition and median wages have been steadily rising at comparable rates with tuition increasing at a slightly faster rate, contributing to affordability issues of higher education. Additionally, significant variation in debt and wages levels around the US indicates geographic location and institutional pricing strategies play a role in shaping student debt outcomes across income brackets.